In [7]:
import pandas as pd

exp = pd.read_csv("../data/processed/exports_clean_2025_07_to_2025_12.csv")
prod_archive = pd.read_csv("../data/processed/production_clean_archive_2025_07_to_2025_12.csv")
prod_latest = pd.read_csv("../data/processed/production_clean_latest_2025_07_to_2025_12.csv")

print("Exports shape:", exp.shape)
print("Production archive shape:", prod_archive.shape)
print("Production latest shape:", prod_latest.shape)

print("\nExports columns:")
print(exp.columns.tolist())

print("\nProduction archive columns:")
print(prod_archive.columns.tolist())

print("\nProduction latest columns:")
print(prod_latest.columns.tolist())

Exports shape: (826, 14)
Production archive shape: (15372, 17)
Production latest shape: (7704, 17)

Exports columns:
['release_month', 'report_month', 'year', 'quarter', 'destination', 'metric_name', 'product', 'metric_group', 'unit', 'period_type', 'report_scope', 'is_cumulative', 'tonnes', 'source_file']

Production archive columns:
['release_month', 'date', 'quarter', 'quarter_start_date', 'quarter_end_date', 'year', 'product', 'metric_group', 'unit', 'period_type', 'measure', 'animal', 'state', 'series_type', 'series_id', 'value', 'source_file']

Production latest columns:
['release_month', 'date', 'quarter', 'quarter_start_date', 'quarter_end_date', 'year', 'product', 'metric_group', 'unit', 'period_type', 'measure', 'animal', 'state', 'series_type', 'series_id', 'value', 'source_file']


In [8]:
archive_dupes = prod_archive.duplicated(subset=["date", "series_id"]).sum()
latest_dupes = prod_latest.duplicated(subset=["date", "series_id"]).sum()

print("Archive duplicate count:", archive_dupes)
print("Latest duplicate count:", latest_dupes)

Archive duplicate count: 7668
Latest duplicate count: 0


In [9]:
print(prod_latest[["date", "quarter", "quarter_start_date", "quarter_end_date"]].drop_duplicates().sort_values("date").tail(10))

            date quarter quarter_start_date quarter_end_date
7344  2023-09-01  2023Q3         2023-07-01       2023-09-30
7380  2023-12-01  2023Q4         2023-10-01       2023-12-31
7416  2024-03-01  2024Q1         2024-01-01       2024-03-31
7452  2024-06-01  2024Q2         2024-04-01       2024-06-30
7488  2024-09-01  2024Q3         2024-07-01       2024-09-30
7524  2024-12-01  2024Q4         2024-10-01       2024-12-31
7560  2025-03-01  2025Q1         2025-01-01       2025-03-31
7596  2025-06-01  2025Q2         2025-04-01       2025-06-30
7632  2025-09-01  2025Q3         2025-07-01       2025-09-30
7668  2025-12-01  2025Q4         2025-10-01       2025-12-31


In [10]:
print(exp[["release_month", "report_month", "quarter", "report_scope", "is_cumulative"]].drop_duplicates().sort_values("report_month"))

    release_month report_month quarter  report_scope  is_cumulative
0         2025-07   2025-07-01  2025Q3  monthly_flow              0
138       2025-08   2025-08-01  2025Q3  monthly_flow              0
271       2025-09   2025-09-01  2025Q3  monthly_flow              0
409       2025-10   2025-10-01  2025Q4  monthly_flow              0
553       2025-11   2025-11-01  2025Q4  monthly_flow              0
693       2025-12   2025-12-01  2025Q4  monthly_flow              0


In [11]:
monthly_counts = (
    exp.groupby("report_month")
    .size()
    .reset_index(name="row_count")
    .sort_values("report_month")
)

print(monthly_counts)

  report_month  row_count
0   2025-07-01        138
1   2025-08-01        133
2   2025-09-01        138
3   2025-10-01        144
4   2025-11-01        140
5   2025-12-01        133


In [12]:
print(prod_latest["product"].value_counts())
print(prod_latest["metric_group"].value_counts())
print(prod_latest["unit"].value_counts())
print(sorted(prod_latest["state"].dropna().unique()))


print(exp["product"].value_counts())
print(exp["metric_group"].value_counts())
print(exp["unit"].value_counts())
print(exp["report_scope"].value_counts())
print(exp["destination"].nunique())

product
beef    3852
lamb    3852
Name: count, dtype: int64
metric_group
production    3852
slaughter     3852
Name: count, dtype: int64
unit
tonnes    3852
head      3852
Name: count, dtype: int64
['Australia', 'Australian Capital Territory', 'New South Wales', 'Northern Territory', 'Queensland', 'South Australia', 'Tasmania', 'Victoria', 'Western Australia']
product
all_meat    296
beef        283
lamb        247
Name: count, dtype: int64
metric_group
export_volume    826
Name: count, dtype: int64
unit
tonnes    826
Name: count, dtype: int64
report_scope
monthly_flow    826
Name: count, dtype: int64
53


In [13]:
prod_summary = (
    prod_latest.groupby(["quarter", "product", "metric_group", "unit"])["value"]
    .sum()
    .reset_index()
    .sort_values(["quarter", "product", "metric_group"])
)

print(prod_summary)


exp_summary = (
    exp.groupby(["report_month", "product"])["tonnes"]
    .sum()
    .reset_index()
    .sort_values(["report_month", "product"])
)

print(exp_summary)

    quarter product metric_group    unit      value
0    1972Q3    beef   production  tonnes   721276.0
1    1972Q3    beef    slaughter    head     3425.6
2    1972Q3    lamb   production  tonnes   144670.0
3    1972Q3    lamb    slaughter    head     9454.3
4    1972Q4    beef   production  tonnes   631850.0
..      ...     ...          ...     ...        ...
851  2025Q3    lamb    slaughter    head    10404.0
852  2025Q4    beef   production  tonnes  1425777.0
853  2025Q4    beef    slaughter    head     4585.0
854  2025Q4    lamb   production  tonnes   273882.0
855  2025Q4    lamb    slaughter    head    11519.9

[856 rows x 5 columns]
   report_month   product         tonnes
0    2025-07-01  all_meat  371266.163854
1    2025-07-01      beef  250577.492879
2    2025-07-01      lamb   45664.012133
3    2025-08-01  all_meat  326147.339406
4    2025-08-01      beef  222024.017856
5    2025-08-01      lamb   35325.969816
6    2025-09-01  all_meat  342953.087168
7    2025-09-01      bee